# ONS Data Joins

This notebook details join steps with ONS (Office for National Statistics) data.

By joining the FHRS data to this location-focused ONS data, area factors can be used to help predict the relative risk of a given establisment.  

Two open data files have been downloaded and saved to `../data/raw/`:

__[Lower Layer Super Output Areas](https://open-geography-portalx-ons.hub.arcgis.com/datasets/postcode-to-oa-2021-to-lsoa-to-msoa-to-lad-november-2025-best-fit-lookup-in-the-uk)__ - "A best-fit lookup between postcodes, 2021 Census Output Areas (OA), Lower Layer Super Output Areas (LSOA), Middle Layer Super Output Areas (MSOA) and local authority districts (LAD). Postcodes are as at November 2025 in the UK and are best-fitted by plotting the location of the postcode's mean address into the areas of the output geographies." **File size is approx. 416MB**

__[English indices of deprivation 2025](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025)__ - "Statistics on relative deprivation in small areas in England" **File size is approx. 2MB**

### Load, Check and prepare FSA and LSOA data for joins

First, I need to load the cleaned FSA data, followed by the LSOA postcode lookup data, checking both load steps have completed ok.  

In [1]:
# load and check cleaned FSA data:
import pandas as pd

df = pd.read_csv(
    "../data/processed/fsa_london_establishments_clean.csv",
    dtype={"PostCode": str, "postcode_tier": str} # prevent CSV mis-read on postcode fields
)

print(df.shape)
df.head()

(81217, 28)


,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,FHRSID,LocalAuthorityBusinessID,...,RatingKey,RatingValue,RightToReply,SchemeType,geocode.longitude,geocode.latitude,scores.Hygiene,scores.Structural,scores.ConfidenceInManagement,postcode_tier
0,NaN,309 Wood Lane,NaN,Dagenham,5 Elms Cafe,Restaurant/Cafe/Canteen,1,0,1714830,81398,...,fhrs_5_en-gb,5,NaN,FHRS,0.142421,51.554493,5.0,5.0,5.0,full
1,NaN,446 Becontree Avenue,NaN,Dagenham,7 TILL 11,Retailers - other,4613,0,115956,45140,...,fhrs_5_en-gb,5,NaN,FHRS,0.129392,51.558435,5.0,5.0,5.0,full
2,NaN,460 Lodge Avenue,NaN,Dagenham,Aafio Mini Market,Retailers - other,4613,0,1413686,77594,...,fhrs_5_en-gb,5,NaN,FHRS,0.110646,51.534718,5.0,5.0,0.0,full
3,NaN,North Street,NaN,Barking,Abbey Children's Centre Day Nursery,Caring Premises,5,0,128460,58427,...,fhrs_5_en-gb,5,NaN,FHRS,0.075250,51.541375,5.0,5.0,5.0,full
4,NaN,1 Hewett Road,NaN,Dagenham,Abbey Kebab & Pizza,Takeaway/sandwich shop,7844,0,122900,5136,...,fhrs_5_en-gb,5,NaN,FHRS,0.128666,51.550930,5.0,5.0,5.0,full


In [2]:
# load LSOA lookup file into memory

import pandas as pd

# only two columns are needed from this otherwise v large file
lsoa_lookup = pd.read_csv(
    "../data/raw/postcode_lsoa_lookup.csv",
    usecols=["pcds", "lsoa21cd"],
        # this approach is more efficient than loading the lot and discarding other columns
    dtype=str, # force string (lookup only data but it may look like numeric etc.)
        # and pandas likes to misinterpret code-like columns otherwise
)

# check load ok
print(lsoa_lookup.shape)
print(lsoa_lookup.head())

(2720556, 2)
      pcds   lsoa21cd
0  AB1 0AA  S01013490
1  AB1 0AB  S01013490
2  AB1 0AD  S01013490
3  AB1 0AE  S01013856
4  AB1 0AF  S01013487


-> note the `pcds` column:

Spaces are still preseent in the postcode column. As I will be joining on an exact string-to-string match, spacing between the two join fields needs to be consistent.  

I believe the cleanest approach will be to strip out all spaces, as they are no longer required for classifying postcode format / tiers:

In [3]:
# create new, formatted postcode column in lookup table:
lsoa_lookup["postcode_key"] = lsoa_lookup["pcds"].str.replace(" ", "", regex=False)

# create equivalent column in FSA table:
df["postcode_key"] = df["PostCode"].str.upper().str.strip().str.replace(" ", "", regex=False)

# check resulting outputs have no spaces:
print(lsoa_lookup["postcode_key"].head())
print(df["postcode_key"].head())

0    AB10AA
1    AB10AB
2    AB10AD
3    AB10AE
4    AB10AF
Name: postcode_key, dtype: str
0     RM83NH
1     RM83UB
2     RM94QS
3    IG118JA
4     RM82XT
Name: postcode_key, dtype: str


-> both outputs are of a matching 'unspaced' format and are ready to join.  

### Join FSA data - LSOA Lookup data

As the FSA data has different 'tiers' of postcode completeness, different approaches are requied for each when joining to the full (outward & inward code) postcodes in the lookup data.

First I will handle the full tier join:

In [ ]:
# DF with full tier postcodes only
full_df = df[df["postcode_tier"] == "full"].copy()
    # .copy() ensures explicit, independent copy of filtered DF

# left join to keep all full_df rows
joined = full_df.merge(
    lsoa_lookup[["postcode_key", "lsoa21cd"]], # required columns only
    on="postcode_key",
    how="left",
)

# inspect join results
print(f"Rows going in: {len(full_df)}")
print(f"Rows coming out: {len(joined)}")

print(f"Matched (lsoa21cd field not null): {joined['lsoa21cd'].notna().sum()}")
print(f"Unmatched: {joined['lsoa21cd'].isna().sum()}")

Rows going in: 70488
Rows coming out: 70488
Matched (lsoa21cd not null): 70422
Unmatched: 66


-> 99.9% of rows joined ok, with no duplicate matches (rows in = rows out)

An encouraging (albeit not unexpected) hit rate.  

In terms of the 66 unmatched rows:

In [ ]:
# select & inspect unmatched rows:
unmatched_postcodes = joined.loc[joined["lsoa21cd"].isna(), ["PostCode", "postcode_key", "LocalAuthorityName", "RatingDate"]]
    # .loc uses two-part syntax [rows, columns], allows for simple & clean operation
    # filters rows, selects columns, prevents SettingWithCopyWarning from

print(unmatched_postcodes)

       PostCode postcode_key    LocalAuthorityName           RatingDate
505    RM10 6NH      RM106NH  Barking and Dagenham  2024-09-20T00:00:00
1523    NW2 8BB       NW28BB                Barnet  2025-10-14T00:00:00
3151    NW2 8BA       NW28BA                Barnet  1901-01-01T00:00:00
3203    NW9 7GQ       NW97GQ                Barnet  1901-01-01T00:00:00
3373    DA5 3TB       DA53TB                Bexley  2025-09-22T00:00:00
...         ...          ...                   ...                  ...
61344   E10 4QD       E104QD        Waltham Forest  2024-07-10T00:00:00
65397   W1J 7FH       W1J7FH           Westminster  2024-04-25T00:00:00
67160  SW1E 6DB      SW1E6DB           Westminster  2023-01-12T00:00:00
68445   W1K 2DA       W1K2DA           Westminster  2026-08-07T00:00:00
68557   W1K 6JV       W1K6JV           Westminster  2026-05-08T00:00:00

[66 rows x 4 columns]


-> resulting `PostCode` / `postcode_key` values _appear_ to be legitimate postcodes, despite their being unmatched with the lookup data.  

Plenty of `RatingDate` values pre-date the latest update (2025) to the lookup table, so newly-added postcodes is not an explanation here. Dates from 1901-01-01 are evidently placeholders where no visit has taken place, rather than a concern (or actual inspection date...!).    

There is a spread across boroughs, which points to actual edge cases rather than a consistent administrative issue/quirk.  

A quick search for some of the postcodes shows that many of the suffixes do not point to an existing inward code for a given outward code, which likely indcates a typo or inputting error-type issue, rather than an issue stemming from the join or the LSOA lookup data.  

As there is no dominant pattern to explain the unmatched codes, and as the unmatched rows account for less than 0.1% of the full postcode dataset, I will exclude these rows from LSOA-dependent deprivation factor analysis.  

I'll now merge the full-tier join results back into the main FSA DataFrame. This can be achieved by merging the main, unfiltered DataFrame against the lookup, rather than trying to stitch the newly-created `joined` DF back in. Full-tier rows should once again match, and partial (outward or sector tier) should not match, falling through without the need for filtering.  

While this is effectively the same join again, the steps just taken have not been wasted as they allowed for validation of the join, in isolation. 

In [ ]:
# merge the entire main DF against lsoa_lookup
    # only full-tier matches should succeed
    # as outward/sector rows are too short for a full match
df = df.merge(
    lsoa_lookup[["postcode_key", "lsoa21cd"]],
    on="postcode_key",
    how="left",
)

print(df.shape)
print(df["lsoa21cd"].notna().sum())

print(df.groupby("postcode_tier")["lsoa21cd"].apply(lambda x: x.notna().sum()))

postcode_tier
blank               0
full            70422
outward             0
sector              0
unclassified        1
Name: lsoa21cd, dtype: int64


In [8]:
matched_unclassified = df[(df["postcode_tier"] == "unclassified") & df["lsoa21cd"].notna()]
print(matched_unclassified[["PostCode", "postcode_key", "lsoa21cd"]])

      PostCode postcode_key   lsoa21cd
30361  E8 2 NP        E82NP  E01035641


In [9]:
df.loc[(df["postcode_tier"] == "unclassified") & df["lsoa21cd"].notna(), "postcode_tier"] = "full"

print(df["postcode_tier"].value_counts())

postcode_tier
full            70489
outward          8504
sector           1113
blank            1094
unclassified       17
Name: count, dtype: int64
